In [1]:
import os, sys
cur_directory = os.getcwd()
project_folder = os.path.abspath(os.path.join(cur_directory, os.pardir, os.pardir))
save_directory = os.path.join(cur_directory, 'saved')
sys.path.append(os.path.join(project_folder, 'analysis-suite'))

In [93]:
import numpy as np
from numpy import linalg as la
import matplotlib as mpl
from matplotlib import pyplot as plt
import analyses as als
import datasets as ds
import models as mds
import tools
from random import sample
from tqdm import tqdm
from argparse import Namespace
from time import gmtime, strftime
import pickle

In [94]:
mov_ind = 4

In [95]:
data_path = os.path.join(project_folder, 'data', 'nat_scenes_data')
D = ds.DataNatScenes(data_path, mov_ind=mov_ind)
print(f'current stimulus is {D.movname}')
mov_ind = D.mov_ind

current stimulus is opticflow


# Train/test validation - epochs

Test optimum seems to be at 5-10

In [97]:
V_width = 7
V_E_buffer = 0
E_width = 1

N_training_repeats = 10
# iterations_array = np.array([1, 5, 10, 15, 20, 30])
cg_params = {
    "order": 1,
    "output_dim": 200
}

critic_params = {
    "hidden_dim": 32,
    "embed_dim": 30,
    "layers": 2,
    "activation": "tanh",
}

opt_params = {"track_quantities": False, "quiet": True, "iterations": 10}
# opt_params = {"track_quantities": False, "quiet": True}
mi_measurement_params = {'quiet':True, 'batch_size':500, 'N_batches':500}


train_fraction = 0.5
samp_keep = 1

## Prepare data

In [98]:
D.reset()
D.trim(20)
V, E = D.create_V_and_E_datasets(V_width, V_E_buffer, E_width, samp_keep=samp_keep)

In [99]:
def expt(swap):
    if swap:
        IB_func = als.IB_swap_xy
        MI_func = als.MI_NCE_swap_xy
    else:
        IB_func = als.IB
        MI_func = als.MI_NCE

    coarse_grainer = mds.PolynomialLayer(**cg_params)
    critic_function = mds.SeparableCritic(**critic_params)

    # make training and test subsets
    V.reset()
    E.reset()
    current_train_points, current_test_points = tools.generate_train_test_indices(
        V.P, train_fraction
    )
    V.subset_points(current_train_points)
    E.subset_points(current_train_points)
    train_points = current_train_points
    test_points = current_test_points

    # Train current
    IB_func(
        X=V.X,
        Y=E.X,
        coarse_grainer=coarse_grainer,
        critic_function=critic_function,
        **opt_params
    )

    # Measure InfoNCE for train
    T_train = coarse_grainer(V.X.T).numpy().T
    mi_train = MI_func(
        X=T_train, Y=E.X, critic_function=critic_function, **mi_measurement_params
    )
    mi_averages_train = mi_train["mi"]
    mi_stdevs_train = mi_train["mi_stdev"]

    # Measure InfoNCE for test
    V.reset()
    E.reset()
    V.subset_points(current_test_points)
    E.subset_points(current_test_points)
    T_test = coarse_grainer(V.X.T).numpy().T
    mi_test = MI_func(
        X=T_test, Y=E.X, critic_function=critic_function, **mi_measurement_params
    )
    mi_averages_test = mi_test["mi"]
    mi_stdevs_test = mi_test["mi_stdev"]

    coarse_grainer_array = coarse_grainer
    critic_function_array = critic_function
    
    return Namespace(**locals())

In [100]:
swap = [expt(True) for n in range(N_training_repeats)]
ctl = [expt(False) for n in range(N_training_repeats)]

In [101]:
def extract(expt_list, key):
    dict_array = [vars(a) for a in expt_list]
    B = np.array([b[key] for b in dict_array])
    return B

In [102]:
swap_mi = extract(swap, "mi_averages_test")
ctl_mi = extract(ctl, "mi_averages_test")

swap_mean = np.mean(swap_mi)
ctl_mean = np.mean(ctl_mi)
swap_mean, ctl_mean

(1.0569426936149597, 1.0402186702728273)

In [103]:
swap_mi = extract(swap, "mi_averages_train")
ctl_mi = extract(ctl, "mi_averages_train")

swap_mean = np.mean(swap_mi)
ctl_mean = np.mean(ctl_mi)
swap_mean, ctl_mean

(1.370386576652527, 1.389412833213806)

In [104]:
swap_std = np.sqrt(np.var(swap_mi))
ctl_std = np.sqrt(np.var(ctl_mi))
swap_std, ctl_std

(0.007031294328276362, 0.010872126632655464)

In [41]:
swap.mi_averages_test, ctl.mi_averages_test

AttributeError: 'list' object has no attribute 'mi_averages_test'